<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EC%9D%91%EC%9A%A9_7%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

문항 1. GAN의 경쟁적 학습 구조 GAN(Generative Adversarial Networks)은 두 개의 네트워크가 서로 경쟁하며 학습합니다.

랜덤 노이즈를 입력받아 실제와 유사한 이미지를 만들어내려는 네트워크 ( A )

입력된 이미지가 실제 데이터인지 A가 만든 가짜 데이터인지 구별하려는 네트워크 ( B ) 위 (A)와 (B)에 들어갈 명칭을 적으세요.

답변 작성란:

( A ): Generator

( B ): Discriminator

문항 2. VAE(Variational Autoencoder)의 잠재 공간
GAN과 달리 VAE는 입력 데이터를 잠재 공간(Latent Space)의 **( A )** 으로 매핑하여 학습합니다. 이를 통해 잠재 변수 z를 샘플링할 때, 평균과 분산을 이용하는 ( B ) 트릭(Trick)을 사용하여 미분 가능하게 만듭니다. 빈칸에 들어갈 말은 무엇입니까?

답변 작성란:

( A ) (특정 수학적 개념): 압축된 확률적 표현

( B ) (기법 이름): 재매개변수화

문항 3. DCGAN 생성자(Generator) 구현 DCGAN의 생성자는 랜덤 벡터(z)를 입력받아 전치 합성곱(Transposed Convolution)을 통해 이미지 크기를 키워 나갑니다. 아래 코드에서 ConvTranspose2d를 사용하여 특징 맵의 크기를 2배로 키우는 층을 정의하세요.

In [ ]:
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, latent_dim, img_channels):
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            # 입력: latent_dim (100) -> 특징 맵 생성
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            # TODO: 특징 맵 크기 2배 확장 (512 -> 256 채널)
            # kernel_size=4, stride=2, padding=1 설정을 사용
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=Flase),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            # 최종 출력 층: Tanh 활성화 함수 사용 (이미지 픽셀 범위 -1 ~ 1)
            nn.ConvTranspose2d(128, img_channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z)

문항 4. DCGAN 판별자(Discriminator) 구현 판별자는 이미지를 입력받아 진짜(1)인지 가짜(0)인지 확률을 출력합니다. DCGAN 논문에 따르면 판별자에서는 활성화 함수로 ReLU 대신 **( A )** 를 사용하여 기울기 소실 문제를 완화합니다. 이를 적용하여 코드를 완성하세요.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_channels):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            # 입력 이미지 -> 특징 추출
            nn.Conv2d(img_channels, 128, 4, 2, 1, bias=False),

            # TODO: LeakyReLU 활성화 함수 적용 (negative_slope=0.2)
            nn.LeakyReLU(0.2)

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2)

            # ... (중간 층 생략) ...

            # 최종 출력: 확률값 (0~1)
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.model(img)

문항 5. 가중치 초기화 (Weight Initialization) DCGAN 논문에서는 모델의 학습 안정성을 위해 모든 가중치를 평균 0, 표준편차 0.02의 정규분포로 초기화할 것을 권장합니다. 이를 수행하는 함수를 완성하세요.

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    # 합성곱(Conv) 또는 전치 합성곱(ConvTranspose) 층인 경우
    if classname.find('Conv') != -1:
        # TODO: 가중치(weight)를 평균 0.0, 표준편차 0.02로 정규분포 초기화
        nn.init.normal_(m.weight.data, 0.0, 0.02)

    # 배치 정규화(BatchNorm) 층인 경우
    elif classname.find('BatchNorm') != -1:
        # 가중치는 1.0, 편향(bias)은 0.0으로 초기화
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        # TODO: 편향(bias)을 0으로 상수 초기화
        nn.init.constant_(m.bias.data , 0)

문항 6. 적대적 학습 루프 (Training Loop) GAN의 학습은 판별자(D)와 생성자(G)를 번갈아 가며 업데이트합니다. 아래 코드는 판별자를 학습시키는 단계입니다. 진짜 이미지와 가짜 이미지에 대한 손실을 각각 계산하여 합산하는 코드를 작성하세요.

(손실 함수 criterion은 BCELoss라고 가정합니다.)

In [ ]:
# real_imgs: 실제 이미지 배치
# fake_imgs: 생성자가 만든 가짜 이미지 배치
# real_label: 1로 채워진 텐서
# fake_label: 0으로 채워진 텐서

# ---------------------
#  Train Discriminator
# ---------------------
optimizer_D.zero_grad()

# 1. 진짜 이미지에 대한 손실 계산
pred_real = discriminator(real_imgs)
# TODO: 진짜 이미지와 real_label 간의 손실 계산
loss_real = criterion(pred_real, real_label)

# 2. 가짜 이미지에 대한 손실 계산
pred_fake = discriminator(fake_imgs.detach()) # G의 그래디언트 전파 방지
# TODO: 가짜 이미지와 fake_label 간의 손실 계산
loss_fake = criterion(pred_fake, fake_label)

# 3. 역전파 및 가중치 업데이트
loss_D = (loss_real + loss_fake) / 2
loss_D.backward()
optimizer_D.step()